# Phase 11 — Stage 1: SFT Training with SFTTrainer
## CodeMentor-LLM
Fine-tuning meta-llama/Llama-3.2-3B-Instruct
on CodeAlpaca-20K dataset using QLoRA + SFTTrainer.

## Training Config
- Model     : meta-llama/Llama-3.2-3B-Instruct
- Dataset   : Abdulmoiz123/codementor-llm-splits
- Method    : SFT + QLoRA (4-bit NF4)
- Epochs    : 3
- LR        : 2e-4
- Batch size: 4
- Tracking  : Weights & Biases

# Libraries

In [ ]:
!pip uninstall wandb numpy -y
!pip install numpy==1.26.4
!pip install transformers==4.49.0 peft==0.14.0 trl==0.15.2 bitsandbytes==0.45.3 accelerate==1.5.1 datasets==3.3.2 wandb

# HF and WandB Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import wandb

# Login to HuggingFace
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("HuggingFace login successful")

# Login to W&B
wandb_api_key = userdata.get('WANDB_API_KEY')
wandb.login(key=wandb_api_key)
print("W&B login successful")

#  Check GPU

In [ ]:
import torch

print(f"GPU Name     : {torch.cuda.get_device_name(0)}")
print(f"Total Memory : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"CUDA Version : {torch.version.cuda}")
print(f"PyTorch      : {torch.__version__}")

# Load model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded successfully")

# Load model
print("\nLoading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Model loaded — {model.get_memory_footprint() / 1024**3:.2f} GB")

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)
print("Model prepared for k-bit training")

# Apply LoRA adapter

In [ ]:
# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load dataset


In [ ]:
from datasets import load_dataset

# Load train and validation splits
print("Loading dataset splits...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-splits")

train_dataset = dataset["train"].select(range(5000))
val_dataset = dataset["validation"].select(range(1000))

print(f"Train samples      : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"\nSample:")
print(train_dataset[0]["text"])

#  Initialize W&B run

In [ ]:
import wandb

# Initialize W&B run
wandb.init(
    project="codementor-llm",
    name="sft-llama3.2-3b-codealapaca",
    config={
        "model": "meta-llama/Llama-3.2-3B-Instruct",
        "dataset": "Abdulmoiz123/codementor-llm-splits",
        "epochs": 3,
        "learning_rate": 2e-4,
        "batch_size": 4,
        "gradient_accumulation": 4,
        "lora_r": 16,
        "lora_alpha": 32,
        "quantization": "4-bit NF4",
    }
)

print("W&B run initialized successfully")
print(f"Run name: {wandb.run.name}")
print(f"Run URL: {wandb.run.url}")

# Define Training Arguments

In [ ]:
from trl import SFTTrainer, SFTConfig

# SFT Config
sft_config = SFTConfig(
    output_dir="./checkpoints/sft",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    push_to_hub=True,
    hub_model_id="Abdulmoiz123/codementor-llm-sft",
    hub_strategy="every_save",
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    max_seq_length=2048,
    dataset_text_field="text",
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
)

print("SFTTrainer initialized successfully")
print(f"Training samples : {len(train_dataset)}")
print(f"Eval samples     : {len(val_dataset)}")

#  Start training

In [ ]:
# Start training
trainer.train()

print("\nTraining complete!")

# Save final model

In [ ]:
# Save final model to HF Hub
trainer.push_to_hub()
print("Final model pushed to HuggingFace Hub successfully")
print(f"Model available at: https://huggingface.co/Abdulmoiz123/codementor-llm-sft")

# W&B finish

In [ ]:
# Finish W&B run
wandb.finish()
print("W&B run finished successfully")